# Exercise 1: Linear and Logistic Regression with Error Estimation


## Instructions






### Setup check

Run the cell below before you begin. Every entry should report `found`.

In [ ]:
import os
import sys

print("Python", sys.version.split()[0])

for package in ["numpy", "pandas", "matplotlib", "seaborn", "sklearn"]:
    try:
        __import__(package)
        print(f"{package:<12} found")
    except ImportError:
        print(f"{package:<12} MISSING  ->  pip install {package}")

for filename in ["headbrain.csv", "titanic.csv", "diabetes.csv"]:
    path = os.path.join("datasets", filename)
    status = "found" if os.path.isfile(path) else "MISSING" 
    print(f"{filename:<16} {status}")

---
# Part A. Linear regression

## Program 1. Ordinary least squares regression, from first principles and with `scikit-learn`

### Aim

To implement simple linear regression in Python without a machine learning library, to reproduce
the same fit using `scikit-learn`, and to attach an uncertainty estimate to the fitted
coefficients.

### Theory

The regression line is written as $y = b_0 + b_1 x$. The least squares estimates of the two
coefficients are

$$
\newcommand{\vect}{\mathbf}
\begin{align}
b_1 &= \frac{\displaystyle \sum_{i=1}^{m}(x_i-\bar{x})(y_i-\bar{y})}{\displaystyle \sum_{i=1}^{m}(x_i-\bar{x})^2}
= \frac{(\vect{x}-\bar{x})^T(\vect{y}-\bar{y})}{(\vect{x}-\bar{x})^T(\vect{x}-\bar{x})}
\\
b_0 &= \bar{y}-b_1\bar{x}
\end{align}
$$

The quality of the fit is summarised by

$$
\begin{align}
\text{RMSE} &= \sqrt{\frac{1}{m}\sum_{i=1}^{m}(y_i-\hat{y_i})^2}
= \sqrt{\frac{(\vect{y}-\hat{\vect{y}})^T(\vect{y}-\hat{\vect{y}})}{m}}
\\
\text{SS}_{\text{tot}} &= \sum_{i=1}^{m}(y_i - \bar{y})^2 = (\vect{y}-\bar{y})^T(\vect{y}-\bar{y})
\\
\text{SS}_{\text{res}} &= \sum_{i=1}^{m}(y_i - \hat{y}_i)^2 = (\vect{y}-\hat{\vect{y}})^T(\vect{y}-\hat{\vect{y}})
\\
R^{2} &= 1-\frac{\text{SS}_{\text{res}}}{\text{SS}_{\text{tot}}}
\end{align}
$$

where $\vect{x} = (x_1,x_2,\dots,x_m)$ is the input vector, $\vect{y} = (y_1,y_2,\dots,y_m)$ is the
output vector, $\bar{x}$ and $\bar{y}$ are the corresponding means, and
$\hat{\vect{y}} = (\hat{y}_1,\hat{y}_2,\dots,\hat{y}_m)$ is the vector of predicted values.

Under the usual assumption of independent errors of constant variance, the standard errors of the
two coefficients follow from the residual variance
$s^2 = \text{SS}_{\text{res}}/(m-2)$:

$$
\begin{align}
\text{SE}(b_1) &= \sqrt{\frac{s^2}{\sum_{i=1}^{m}(x_i-\bar{x})^2}}
\\
\text{SE}(b_0) &= \sqrt{s^2\left[\frac{1}{m} + \frac{\bar{x}^2}{\sum_{i=1}^{m}(x_i-\bar{x})^2}\right]}
\end{align}
$$

The divisor $m-2$ appears because two parameters have already been estimated from the same data.

### TASK A1. Implementing the regression class 

Complete the three methods of the class below. Use vectorised NumPy operations only, without an
explicit Python loop over the samples, and without importing `scikit-learn`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
class LinearRegression:
    """Simple linear regression by ordinary least squares."""

    def fit(self, X, y):
        """Estimate b0 and b1 from one-dimensional arrays X and y.

        Stores the results as self.b0 and self.b1 and returns self.
        """
        m = X.shape[0]

        x_mean = X.mean()
        y_mean = y.mean()

        x_centred = X - x_mean
        y_centred = y - y_mean

        self.b1 = (x_centred @ y_centred) / (x_centred @ x_centred)
        self.b0 = y_mean - self.b1 * x_mean

        print(f"(b0, b1) = ({self.b0:.3f}, {self.b1:.3f})")
        return self

    def predict(self, X):
        """Return the predicted response for the array X."""
        return self.b0 + self.b1 * X

    def evaluate(self, X, y):
        """Print and return the RMSE and the R^2 score as a tuple."""

        y_pred = self.predict(X)

        m = X.shape[0]
        rmse = np.sqrt(((y - y_pred) @ (y - y_pred)) / m)
        ss_tot = (y - y.mean()) @ (y - y.mean())
        ss_res = (y - y_pred) @ (y - y_pred)
        r2 = 1 - ss_res / ss_tot

        print("Root mean squared error:", round(rmse, 3))
        print("R^2 value:", round(r2, 3))
        return (rmse, r2)

### TASK A2. Plotting the fit

Complete the plotting helper. The scatter of the observed data and the fitted straight line must
appear on the same axes.

def regression_plot(X, y, model, title=""):
    """Scatter plot of the data with the fitted regression line drawn over it."""
    plt.figure(figsize=(10, 6))
    plt.title(title)

    plt.xlabel("Head size (cm^3)")
    plt.ylabel("Brain weight (grams)")

    pad = 100
    x_line = np.array([X.min() - pad, X.max() + pad]).reshape(-1, 1)
    y_line = model.predict(x_line)

    plt.scatter(X, y, s=15, label="observed data")
    plt.plot(x_line, np.ravel(y_line), color="red", label="fitted line")
    plt.legend()
    plt.show()

### TASK A3. Fitting the head and brain dataset 

The dataset relates the head size of an adult to the mass of the brain. Load it, extract the two
columns of interest, fit the model, plot the result, and evaluate the fit.

In [ ]:
data = pd.read_csv("datasets/headbrain.csv")
print(data.shape)
data.head()

In [ ]:
X = data["Head Size(cm^3)"].values
y = data["Brain Weight(grams)"].values

In [ ]:
model = LinearRegression().fit(X, y)
regression_plot(X, y, model, title="Brain weight against head size (own implementation)")
model.evaluate(X, y)

### TASK A4. Comparison with `scikit-learn` 

Repeat the fit with the library estimator and confirm that the two implementations agree.

Note that `scikit-learn` expects a two-dimensional array of predictors, so the input has to be
reshaped from `(m,)` to `(m, 1)`.

In [ ]:
from sklearn.linear_model import LinearRegression as SkLinearRegression
from sklearn.metrics import mean_squared_error

X1 = X.reshape(-1, 1)

In [ ]:
sk_model = SkLinearRegression().fit(X1, y)
regression_plot(X1, y, sk_model, title="Brain weight against head size (scikit-learn)")

sk_b0 = sk_model.intercept_
sk_b1 = sk_model.coef_[0]
print("Intercept:", sk_b0)
print("Coefficient:", sk_b1)

y_pred = sk_model.predict(X1)
rmse = np.sqrt(mean_squared_error(y, y_pred))
print("Root mean squared error:", round(rmse, 3))
print("R^2 value:", round(sk_model.score(X1, y), 3))

print("Difference in b0:", abs(model.b0 - sk_b0))
print("Difference in b1:", abs(model.b1 - sk_b1))

### TASK A5. Error estimation 

A coefficient reported without an uncertainty is of limited scientific value. In this task the
uncertainty is estimated in two independent ways, first from the analytical expressions given in
the theory section and then by resampling.

The 95 per cent confidence interval of a coefficient is approximated as
$b \pm t_{0.975,\,m-2}\,\text{SE}(b)$, where the critical value is close to 1.96 for a sample of
this size.

In [ ]:
def coefficient_standard_errors(X, y, model):
    # Return (se_b0, se_b1) for a fitted simple linear regression model.
    m = X.shape[0]
    residuals = y - model.predict(X)
    s2 = (residuals @ residuals) / (m - 2)

    x_centred = X - X.mean()
    sxx = x_centred @ x_centred

    se_b1 = np.sqrt(s2 / sxx)
    se_b0 = np.sqrt(s2 * (1 / m + X.mean() ** 2 / sxx))
    return se_b0, se_b1


se_b0, se_b1 = coefficient_standard_errors(X, y, model)
t_crit = 1.96

print(f"b0 = {model.b0:.3f} +/- {t_crit * se_b0:.3f}   (SE = {se_b0:.3f})")
print(f"b1 = {model.b1:.3f} +/- {t_crit * se_b1:.3f}   (SE = {se_b1:.4f})")

In [ ]:
def bootstrap_slope(X, y, n_resamples=1000, seed=0):
    # Return an array of slope estimates from n_resamples bootstrap replicates.
    rng = np.random.default_rng(seed)
    m = X.shape[0]
    slopes = np.empty(n_resamples)

    for k in range(n_resamples):
        idx = rng.integers(0, m, size=m)
        xs, ys = X[idx], y[idx]
        # slope computed directly here, otherwise fit() would print 1000 times
        xc = xs - xs.mean()
        yc = ys - ys.mean()
        slopes[k] = (xc @ yc) / (xc @ xc)

    return slopes


slopes = bootstrap_slope(X, y)

plt.figure(figsize=(8, 5))
plt.hist(slopes, bins=30)
plt.xlabel("Bootstrap slope estimate")
plt.ylabel("Frequency")
plt.title("Bootstrap distribution of the slope")
plt.show()

print("Bootstrap SD of slope:", round(slopes.std(), 4))
print("Analytical SE(b1):   ", round(se_b1, 4))

# The bootstrap standard deviation (about 0.0132) is very close to the analytical
# SE(b1) of 0.0129. The analytical value rests on the assumption of independent
# errors with constant variance, while the bootstrap makes no such assumption,
# so the close agreement suggests those assumptions hold reasonably well here.

### TASK A6. Written questions 

Answer in the cell below.

1. The fitted slope carries physical units. State them and explain in one sentence what the slope
   means for this dataset.
2. The value of $R^2$ obtained here is well below unity. Give two distinct reasons why a single
   predictor cannot account for all of the variance in the response.
3. Suppose the head size were reported in cubic metres instead of cubic centimetres. State how
   $b_0$, $b_1$, RMSE, and $R^2$ would each change.

ANSWER A6

1. The slope is measured in grams per cubic centimetre (g/cm³) because it shows how head volume relates to brain weight. In this dataset, it means that for every additional 1 cm³ of head volume, the brain weight increases by about 0.263g on average.

2. First, brain weight is affected by more than just head volume. Factors like age, sex, body build, and natural differences between people are not included in the model, so people with the same head size can still have different brain weights. Second, both head volume and brain weight may contain small measurement errors, which also add to the residuals. Because of this, the model explains only the variation shared between head volume and brain weight, which is about 64%.

3. Since 1 m³ = 10⁶ cm³, each x-value is divided by 10⁶. To keep the relationship the same, the slope (b_1) is multiplied by 10⁶, giving about 263,430g/m³. The intercept (b_0) does not change because the y-values are still measured in grams. The RMSE also stays the same since it is measured in grams, and (R^2) remains 0.639 because changing the units of the predictor does not affect it.


---
# Part B. Logistic regression

## Program 1. Preprocessing and classification on the Titanic dataset with `scikit-learn`

### Aim

To prepare a dataset that contains missing values and categorical columns, and to fit and evaluate
a logistic regression classifier on it.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

titanic_df = pd.read_csv("./datasets/titanic.csv")
titanic_df.head()

### TASK B1. Exploratory visualisation 

The helper below draws a single titled figure. Use it to produce the six plots listed in the
comments.

In [ ]:
def plot(title, plot_func, *args, **kwargs):
    plt.figure(figsize=(7, 4))
    plt.title(title)
    plot_func(*args, **kwargs)
    plt.show()


plot("Missing values", sns.heatmap, titanic_df.isnull(), cbar=False, cmap="viridis")
plot("Survival split by sex", sns.countplot, x="Survived", hue="Sex", data=titanic_df)
plot("Survival split by passenger class", sns.countplot, x="Survived", hue="Pclass", data=titanic_df)
plot("Age distribution", plt.hist, titanic_df["Age"].dropna(), bins=30)
plot("Fare distribution", plt.hist, titanic_df["Fare"], bins=40)
plot("Age against passenger class", sns.boxplot, x="Pclass", y="Age", data=titanic_df)

ANSWER B1

Three columns contain missing values: Age (177 missing entries), Cabin (687 missing entries, meaning most passengers have no cabin information), and Embarked (only 2 missing entries).

The box plot shows that the median age decreases as passenger class goes from first to third. First-class passengers are the oldest, with a median age of about 38, followed by second class at around 30, while third-class passengers are the youngest with a median close to 25. This suggests that older passengers were generally wealthier, which is why the missing ages are later filled using the mean age of each passenger class instead of a single overall mean.


### TASK B2. Missing, categorical, and irrelevant columns 

Missing ages are filled with the mean age of the passenger class rather than with the mean of the
whole column, because age and class are correlated. Complete the imputation, then encode the
categorical columns and remove the columns that carry no predictive value.

In [ ]:
mean_ages = {
    p: titanic_df[titanic_df["Pclass"] == p]["Age"].mean()
    for p in sorted(titanic_df["Pclass"].unique())
}
print(mean_ages)


def impute_missing_age(columns):
    age, p_class = columns
    if pd.isnull(age):
        return mean_ages[p_class]
    return age


titanic_df["Age"] = titanic_df[["Age", "Pclass"]].apply(impute_missing_age, axis=1)

plot("Missing values after imputation", sns.heatmap, titanic_df.isnull(), cbar=False, cmap="viridis")

In [ ]:
titanic_df.drop("Cabin", axis=1, inplace=True)
titanic_df.dropna(inplace=True)

sex = pd.get_dummies(titanic_df["Sex"], drop_first=True)
embarked = pd.get_dummies(titanic_df["Embarked"], drop_first=True)
titanic_df = pd.concat([titanic_df, sex, embarked], axis=1)

titanic_df.drop(["Name", "PassengerId", "Ticket", "Sex", "Embarked"], axis=1, inplace=True)
titanic_df.head()

ANSWER B2

`drop_first=True` is used to remove one category from each set of dummy variables, making it the reference or baseline category. This prevents unnecessary duplication of information while keeping all the important information in the data.

If all the indicator columns were retained, the dummy variables for a category would always add up to 1 for every passenger, creating perfect multicollinearity (the dummy variable trap). As a result, the model coefficients would not be uniquely identifiable, making the estimates unstable and difficult to interpret.


### TASK B3. Fitting and evaluating the classifier 

Scale `Age` and `Fare` to the unit interval before fitting, so that the two columns of large
numerical range do not dominate the optimisation. Fix `random_state` in the split so that your
result is reproducible.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import minmax_scale

titanic_df["Age"] = minmax_scale(titanic_df["Age"])
titanic_df["Fare"] = minmax_scale(titanic_df["Fare"])

X = titanic_df.drop("Survived", axis=1)
y = titanic_df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)
predictions = log_model.predict(X_test)

print(classification_report(y_test, predictions))
print(confusion_matrix(y_test, predictions))

# rerunning the split with different seeds, for question 3 of Task B4
for rs in [1, 42, 101]:
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=rs)
    acc = LogisticRegression(max_iter=1000).fit(X_tr, y_tr).score(X_te, y_te)
    print(f"random_state={rs}: accuracy = {acc:.4f}")

### TASK B4. Interpretation 

Answer in the cell below.

1. Write out the confusion matrix you obtained and label the four entries as true positives, false
   positives, true negatives, and false negatives.
2. Define precision and recall in words, and compute both by hand for the survivor class from your
   confusion matrix. Confirm that your values match the classification report.
3. Rerun the split with three different values of `random_state` and record the accuracy each time.
   Comment on the spread and on what it implies about reporting a single accuracy figure.

ANSWER B4

1. With `random_state=42`, the confusion matrix is:

```
[[139  28]
 [ 28  72]]
```

Rows represent the actual classes, while columns represent the predicted classes, with "survived" (1) as the positive class. The 139 in the top-left are true negatives, the 28 in the top-right are false positives, the 28 in the bottom-left are false negatives, and the 72 in the bottom-right are true positives. It just happens that the false positives and false negatives are both 28 for this particular split.

2. Precision measures the proportion of passengers predicted to survive who actually survived. It is calculated as TP / (TP + FP) = 72 / (72 + 28) = 0.72. Recall measures the proportion of actual survivors that the model correctly identified. It is calculated as TP / (TP + FN) = 72 / (72 + 28) = 0.72. These values match the precision and recall for class 1 shown in the classification report.

3. Running the model with different train-test splits gives accuracies of 0.8240 (`random_state=1`), 0.7903 (`random_state=42`), and 0.8165 (`random_state=101`). The difference comes from the different passengers included in each test set, not from changes in the model itself. This is why relying on a single train-test split can be misleading, and using multiple splits or cross-validation gives a more reliable evaluation.


---
## Program 2. Logistic regression by gradient descent, compared with `scikit-learn`

### Aim

To implement logistic regression through gradient descent on the log-likelihood, and to compare its
accuracy with the library implementation on the Pima diabetes dataset.

### Theory

$$
\newcommand{\vect}{\mathbf}
\text{Log-likelihood}, \quad LL = \sum_{i=1}^{m} y_i \log(p_i) + (1-y_i)\log(1-p_i)
$$

where

$$
p_i = p(\vect{x}_i, \vect{w}) = \frac{1}{1+e^{-\vect{w}^T\vect{x}_i}}
$$

Here $\vect{w} = (w_0, w_1, \dots, w_n)$ is the weight vector and
$\vect{x}_i = (1, x_{i1}, x_{i2}, \dots, x_{in})$ is the $i$th input vector, the leading unit
entry accounting for the intercept.

The weights are chosen so that

$$
\max_{\vect{w}} LL = \min_{\vect{w}} (-LL)
$$

The gradient of the negative log-likelihood takes the compact form

$$
\frac{d(-LL)}{d\vect{w}} = \sum_{i=1}^{m}(p_i-y_i)\,\vect{x}_i = (\vect{p}-\vect{y})^T X
$$

so that the update step of gradient descent is

$$
\vect{w} := \vect{w} - \alpha\,(\vect{p}-\vect{y})^T X
$$

where $\alpha$ is the learning rate.

### TASK B5. Implementing the classifier 

Complete the class below. No optimisation routine from a library may be used.

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
class LogitRegression:
    """Binary logistic regression trained by full batch gradient descent."""

    def __init__(self, learning_rate, iterations):
        self.learning_rate = learning_rate
        self.iterations = iterations

    def p(self, X):
        """Return the predicted probability for a design matrix that already carries
        the leading column of ones."""
        return 1 / (1 + np.exp(-(X @ self.w)))

    def fit(self, X, y):
        m, n = X.shape
        X = np.hstack([np.ones((m, 1)), X])
        y = y.squeeze()
        self.w = np.zeros(n + 1)

        for _ in range(self.iterations):
            self.w -= self.learning_rate * ((self.p(X) - y) @ X)

        return self

    def predict(self, X):
        """Return the predicted class label, 0 or 1, for each row of X."""
        m = X.shape[0]
        X = np.hstack([np.ones((m, 1)), X])

        return (self.p(X) >= 0.5).astype(int)

In [ ]:
from sklearn.preprocessing import minmax_scale
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

diabetes_df = pd.read_csv("./datasets/diabetes.csv")
X = minmax_scale(diabetes_df.iloc[:, :-1].values)
y = diabetes_df.iloc[:, -1:].values.reshape(-1)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=1/3, random_state=6
)
print(X_train.shape, X_test.shape)

In [ ]:
def compute_accuracy(model, X_test, y_test):
    y_hat = model.predict(X_test)
    return (y_hat == y_test).mean() * 100


models = [
    LogitRegression(learning_rate=0.1, iterations=1000),
    LogisticRegression(),
]

for model in models:
    model.fit(X_train, y_train)
    acc = compute_accuracy(model, X_test, y_test)
    print(f"{model.__class__.__name__}: test accuracy = {acc:.2f}%")


### TASK B6. Effect of the learning rate 

Train your implementation with learning rates of 0.001, 0.01, 0.1, and 1.0, holding the number of
iterations fixed at 1000. Tabulate the test accuracy against the learning rate and comment in two
or three sentences on what you observe at the two extremes.

In [ ]:
for lr in [0.001, 0.01, 0.1, 1.0]:
    m = LogitRegression(learning_rate=lr, iterations=1000).fit(X_train, y_train)
    print(f"learning rate = {lr:<6} test accuracy = {compute_accuracy(m, X_test, y_test):.2f}%")

ANSWER B6

| Learning rate | Test accuracy |
|-------------| ------------- |
| 0.001         | 80.08%        |
| 0.01          | 76.95%        |
| 0.1           | 74.61%        |
| 1.0           | 73.83%        |

On this train-test split, the accuracy decreases as the learning rate increases. With a learning rate of 0.001, the model has not fully converged after 1000 iterations, but this slight under-training acts like regularisation and gives the best test accuracy, close to scikit-learn's regularised result of 78.91%. In contrast, a learning rate of 1.0 is too large, causing the weights to overshoot the optimum and making the model unstable (even producing an overflow warning in the sigmoid function). In short, a very small learning rate learns too slowly, while a very large one updates too aggressively, leading to poorer performance.
